# 🤖 Phase 2.1: Semantic Classifier (Word2Vec & Logistic Regression)

## 🏗 Overview: The Semantic Representation (Word2Vec)
In this phase, we move beyond simple word counting to **Word Embeddings**. These represent words as dense vectors in a continuous space, where words with similar meanings are positioned closely together.

### 🔎 Rationale: Moving from "Counter" to "Philosopher"
1.  **Word2Vec**: An algorithm that learns the meaning of words by looking at their neighbors. We will train it on our corpus to see how it clusters Fake vs Real concepts.
2.  **Logistic Regression**: We will use the same 'Chef' from our baseline to see how much better it performs when given the 'Semantic Meaning' of the articles instead of just the word counts.

In [ ]:
import os
import pandas as pd
import joblib
import nltk

# Colab Integration: Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # ⚠️ Use exact same folder name as in your Drive
    BASE_PATH = '/content/drive/MyDrive/Project 2/project-nlp-challenge'
    print("✅ Running in Colab. Google Drive mounted.")
except ImportError:
    BASE_PATH = '.'
    print("💻 Running Locally.")

# NLTK Downloads for ephemeral environments
nltk.download('stopwords')
nltk.download('punkt')

# Ensure directories exist
os.makedirs(os.path.join(BASE_PATH, 'models'), exist_ok=True)
os.makedirs(os.path.join(BASE_PATH, 'dataset'), exist_ok=True)


## 1. Data Ingestion
We load our cleaned datasets.

In [ ]:
train_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/train.csv'))
test_df = pd.read_csv(os.path.join(BASE_PATH, 'dataset/test.csv'))
print(f"Data Loaded. Train: {train_df.shape}, Test: {test_df.shape}")

## 2. Word2Vec Embedding Strategy
We train a customized Word2Vec model on our tokenized corpus to map the words into a 100-dimensional semantic space.

In [ ]:
sentences = [str(row).split() for row in train_df['cleaned_text']]
w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=2, workers=4)

# Save the vectorizer
joblib.dump(w2v_model, os.path.join(BASE_PATH, 'models/word2vec_model.joblib'))
print("✅ Word2Vec Model Trained & Saved.")

## 3. Transformation: Averaging Vectors
To represent an entire article, we average the vectors of every word it contains.

In [ ]:
def document_vector(doc, model):
    words = [w for w in str(doc).split() if w in model.wv.index_to_key]
    if not words: return np.zeros(model.vector_size)
    return np.mean(model.wv[words], axis=0)

X_train_w2v = np.vstack(train_df['cleaned_text'].apply(lambda d: document_vector(d, w2v_model)))
X_test_w2v = np.vstack(test_df['cleaned_text'].apply(lambda d: document_vector(d, w2v_model)))
y_train = train_df['label']
y_test = test_df['label']

print("✅ Dense Vectors Generated for documents.")

## 4. Semantic Classifier Training
Training Logistic Regression on our Word2Vec dense vectors.

In [ ]:
lr_w2v_model = LogisticRegression(max_iter=1000)
lr_w2v_model.fit(X_train_w2v, y_train)
joblib.dump(lr_w2v_model, os.path.join(BASE_PATH, 'models/lr_word2vec_classifier.joblib'))
print("✅ Semantic Classifier (Word2Vec + LR) Trained & Saved.")

## 5. Evaluation
How much does 'Meaning' improve accuracy compared to 'Count'?

In [ ]:
y_pred = lr_w2v_model.predict(X_test_w2v)
print(classification_report(y_test, y_pred))